
# ✅ Environment Setup & Fixes (Auto-added)

This notebook cell pins compatible library versions and applies safe import fixes for PyTorch/TorchVision and Transformers tokenizers.  
It prevents the common errors:
- `AttributeError: partially initialized module 'torchvision' has no attribute 'extension'`
- `RuntimeError: operator torchvision::nms does not exist`
- Keras/TensorFlow circular imports
- Tokenizers fork warnings

Run this cell once at the start of the session.


In [ ]:
# --- limit TensorFlow to 1 GPU before importing it ---
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use only first GPU
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

# optional: if GPU gives trouble, uncomment next line to force CPU
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPUs visible:", tf.config.list_physical_devices("GPU"))


In [ ]:
# =========================
# CELL 1 — AUDIT & CLEAN
# =========================
import os, re, json, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import librosa, soundfile as sf
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ----------- CONFIG -----------
BASE_DATASET_PATH = "/kaggle/input/cleaned-combined-fraud"
CSV_NAME          = "combined_all.csv"
CSV_PATH          = os.path.join(BASE_DATASET_PATH, CSV_NAME)

OUT_ROOT          = Path("/kaggle/working/clean")
OUT_AUDIO_DIR     = OUT_ROOT / "audio"           # we’ll write cleaned audio here
SR_TARGET         = 16000                        # consistent sample rate
MIN_SEC           = 0.50                         # too-short clips dropped
MAX_SEC           = 30.0                         # too-long clips truncated (or dropped if extreme)
TRIM_DB           = 30                           # librosa.effects.trim top_db
PEAK_TARGET       = 0.98                         # peak normalization target
ALLOW_EMPTY_TEXT  = True                         # keep rows with empty text
RNG_SEED          = 42

# ----------- 1) Load CSV -----------
df = pd.read_csv(CSV_PATH)

# Standardize label → label_num
df["label"] = df["label"].astype(str).str.strip()
label_map = {
    "Fraudulent":1, "fraudulent":1, "Fraud":1, "Fraudulent ":1,
    "Non_Fraudulent":0, "Non_Fraud":0, "non_fraudulent":0, "Normal":0, "normal":0
}
df["label_num"] = df["label"].map(lambda x: label_map.get(x, 1 if x.lower().startswith("fraud") else 0))

# Resolve audio path (ignore spectrogram images; we won’t use them)
if "audio_path" in df.columns:
    df["audio_path"] = df["audio_path"].apply(lambda p: os.path.join(BASE_DATASET_PATH, str(p)))
else:
    raise ValueError("CSV must have 'audio_path' column pointing to original audio files.")

# Optional transcript column
if "transcription" not in df.columns:
    df["transcription"] = ""

# Drop rows with missing audio
exists = df["audio_path"].apply(os.path.exists)
missing = (~exists).sum()
if missing:
    print(f"⚠️ Dropping {missing} rows with missing audio files")
df = df[exists].reset_index(drop=True)

print("Loaded rows:", len(df))
print(df[["audio_path","label","label_num"]].head(3))

# ----------- 2) Helpers -----------
def clean_text(t: str) -> str:
    t = str(t)
    t = t.lower().strip()
    # remove non-printable
    t = re.sub(r"[^\x20-\x7E]+", " ", t)
    # collapse whitespace
    t = re.sub(r"\s+", " ", t).strip()
    return t

def peak_normalize(y, target_peak=0.98):
    if len(y) == 0:
        return y
    peak = np.max(np.abs(y))
    if peak > 0:
        scale = target_peak / peak
        y = y * min(scale, 10.0)  # safety clamp
    return y

def load_and_clean_audio(path, sr_target=SR_TARGET, trim_db=TRIM_DB, min_sec=MIN_SEC, max_sec=MAX_SEC):
    # load
    y, sr = librosa.load(path, sr=sr_target, mono=True)
    # trim silence
    y, _ = librosa.effects.trim(y, top_db=trim_db)
    # duration check
    dur = len(y) / sr_target
    if dur < min_sec:
        return None, 0.0
    # truncate if too long
    if dur > max_sec:
        y = y[: int(max_sec * sr_target)]
        dur = max_sec
    # normalize
    y = peak_normalize(y, PEAK_TARGET)
    return y, dur

# ----------- 3) Clean pass (write cleaned wavs) -----------
np.random.seed(RNG_SEED)
OUT_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

clean_rows = []
bad_rows = 0

for i, row in tqdm(df.iterrows(), total=len(df), desc="Cleaning audio + transcripts"):
    src = row["audio_path"]
    label_num = int(row["label_num"])
    label_dir = "Fraudulent" if label_num == 1 else "Non_Fraudulent"

    # clean audio
    try:
        y, dur = load_and_clean_audio(src)
        if y is None or dur <= 0:
            bad_rows += 1
            continue
        # deterministic file id
        stem = Path(src).stem
        out_rel = f"{label_dir}/{stem}.wav"
        out_path = OUT_AUDIO_DIR / out_rel
        out_path.parent.mkdir(parents=True, exist_ok=True)
        sf.write(out_path, y, SR_TARGET, subtype="PCM_16")
    except Exception as e:
        bad_rows += 1
        continue

    # clean transcript
    t_clean = clean_text(row.get("transcription", ""))
    if (not ALLOW_EMPTY_TEXT) and (len(t_clean) < 3):
        # skip if too short and we don't allow empty
        continue

    clean_rows.append({
        "audio_path_clean": str(out_path),
        "label": "Fraudulent" if label_num == 1 else "Non_Fraudulent",
        "label_num": label_num,
        "transcription_clean": t_clean,
        "duration_sec": round(dur, 3),
        "sr": SR_TARGET
    })

clean_df = pd.DataFrame(clean_rows)
print(f"\n✅ Cleaning done. Kept {len(clean_df)} clips, dropped {bad_rows}.")
print(clean_df.head(5))

OUT_ROOT.mkdir(parents=True, exist_ok=True)
CLEAN_CSV = OUT_ROOT / "clean_dataset.csv"
clean_df.to_csv(CLEAN_CSV, index=False)
print(f"🧾 Saved: {CLEAN_CSV}")


In [ ]:
# =========================
# CELL 2 — SPLITS + PREVIEW
# =========================
import pandas as pd
from sklearn.model_selection import train_test_split

CLEAN_CSV = "/kaggle/working/clean/clean_dataset.csv"
dfc = pd.read_csv(CLEAN_CSV)

# sanity: drop anything missing
dfc = dfc[dfc["audio_path_clean"].apply(os.path.exists)].reset_index(drop=True)

# stratified 75/15/15
df_train, df_tmp = train_test_split(
    dfc, test_size=0.25, stratify=dfc["label_num"], random_state=42
)
df_val, df_test = train_test_split(
    df_tmp, test_size=0.4, stratify=df_tmp["label_num"], random_state=42
)

print(f"Split sizes → train={len(df_train)} | val={len(df_val)} | test={len(df_test)}")

# save split CSVs for downstream training
df_train.to_csv("/kaggle/working/clean/train.csv", index=False)
df_val.to_csv("/kaggle/working/clean/val.csv", index=False)
df_test.to_csv("/kaggle/working/clean/test.csv", index=False)
print("🧾 Saved splits under /kaggle/working/clean/{train,val,test}.csv")

# quick preview stats
def show_stats(name, d):
    print(f"\n{name}: n={len(d)} | fraud={d['label_num'].sum()} | non_fraud={len(d)-d['label_num'].sum()}")
    print("duration (sec):",
          d["duration_sec"].describe()[["min","mean","50%","max"]].to_dict())
    text_nonempty = (d["transcription_clean"].astype(str).str.len() >= 3).mean()
    print(f"text coverage (≥3 chars): {text_nonempty*100:.1f}%")

show_stats("TRAIN", df_train)
show_stats("VAL", df_val)
show_stats("TEST", df_test)

# Note:
# - We will compute MEL-spectrogram arrays (224x224) on-the-fly from audio during training/inference.
# - We will compute MFCC+Δ+Δ²+contrast (133 dims) on-the-fly as well, using SR=16k and consistent padding.


In [ ]:
# # ===============================
# # CELL 3 — BERT fine-tune (clean + Kaggle-safe)
# # ===============================

# # --- Environment setup ---
# !pip -q install -U "transformers==4.41.2" "torch==2.3.1" "torchvision==0.18.1" "scikit-learn==1.6.1"

# import os, json, random, math, warnings
# from pathlib import Path
# warnings.filterwarnings("ignore")
# os.environ["TOKENIZERS_PARALLELISM"] = "false"  # avoid fork warnings

# import numpy as np
# import pandas as pd
# import torch
# from torch import nn
# from torch.utils.data import Dataset, DataLoader
# from sklearn.utils.class_weight import compute_class_weight
# from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
# from transformers import BertTokenizerFast, BertForSequenceClassification, get_linear_schedule_with_warmup

# # -----------------------
# # Config
# # -----------------------
# CLEAN_DIR = Path("/kaggle/working/clean")
# TRAIN_CSV = CLEAN_DIR / "train.csv"
# VAL_CSV   = CLEAN_DIR / "val.csv"
# TEST_CSV  = CLEAN_DIR / "test.csv"
# OUTPUT_DIR = Path("/kaggle/working/bert_finetuned")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# MODEL_NAME = "bert-base-uncased"
# MAX_LENGTH = 128
# BATCH_SIZE = 16
# EPOCHS     = 3
# LR         = 2e-5
# SEED       = 42
# TEXT_COL   = "transcription_clean"
# LABEL_COL  = "label_num"

# torch.manual_seed(SEED)
# np.random.seed(SEED)
# random.seed(SEED)
# device = "cuda" if torch.cuda.is_available() else "cpu"
# print("Device:", device)

# # -----------------------
# # Load filtered datasets
# # -----------------------
# def load_filtered(csv_path):
#     df = pd.read_csv(csv_path)
#     df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
#     df = df[df[TEXT_COL].str.len() >= 3].reset_index(drop=True)
#     df[LABEL_COL] = df[LABEL_COL].astype(int).clip(0,1)
#     return df

# df_train = load_filtered(TRAIN_CSV)
# df_val   = load_filtered(VAL_CSV)
# df_test  = load_filtered(TEST_CSV)
# print(f"train={len(df_train)} val={len(df_val)} test={len(df_test)}")

# # -----------------------
# # Tokenizer & Dataset
# # -----------------------
# tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

# class TextDataset(Dataset):
#     def __init__(self, df, tokenizer, max_len):
#         self.texts = df[TEXT_COL].tolist()
#         self.labels = df[LABEL_COL].tolist()
#         self.tokenizer = tokenizer
#         self.max_len = max_len
#     def __len__(self): return len(self.texts)
#     def __getitem__(self, idx):
#         enc = self.tokenizer(
#             self.texts[idx],
#             truncation=True,
#             padding="max_length",
#             max_length=self.max_len,
#             return_tensors="pt"
#         )
#         return {
#             "input_ids": enc["input_ids"].squeeze(0),
#             "attention_mask": enc["attention_mask"].squeeze(0),
#             "labels": torch.tensor(self.labels[idx], dtype=torch.long),
#         }

# train_ds, val_ds, test_ds = (
#     TextDataset(df_train, tokenizer, MAX_LENGTH),
#     TextDataset(df_val, tokenizer, MAX_LENGTH),
#     TextDataset(df_test, tokenizer, MAX_LENGTH),
# )
# train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
# val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
# test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

# # -----------------------
# # Model & Optimizer
# # -----------------------
# model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
# optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# # Class-weighted loss
# labels_np = np.array(df_train[LABEL_COL])
# cw = compute_class_weight("balanced", classes=np.array([0,1]), y=labels_np)
# criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(device))

# # Scheduler
# total_steps = len(train_loader) * EPOCHS
# scheduler = get_linear_schedule_with_warmup(
#     optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps
# )

# # -----------------------
# # Training Loop
# # -----------------------
# def evaluate(loader):
#     model.eval()
#     all_labels, all_preds, all_probs = [], [], []
#     with torch.no_grad():
#         for batch in loader:
#             batch = {k:v.to(device) for k,v in batch.items()}
#             out = model(**batch)
#             probs = out.logits.softmax(dim=-1)
#             preds = probs.argmax(dim=-1)
#             all_labels.extend(batch["labels"].cpu().numpy())
#             all_preds.extend(preds.cpu().numpy())
#             all_probs.extend(probs[:,1].cpu().numpy())
#     acc = accuracy_score(all_labels, all_preds)
#     prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="binary", zero_division=0)
#     try: auc = roc_auc_score(all_labels, all_probs)
#     except: auc = -1.0
#     return {"accuracy":acc,"precision":prec,"recall":rec,"f1":f1,"auc":auc}

# best_f1, patience, patience_ctr = 0, 2, 0
# for epoch in range(EPOCHS):
#     model.train()
#     total_loss = 0
#     for batch in train_loader:
#         batch = {k:v.to(device) for k,v in batch.items()}
#         optimizer.zero_grad()
#         out = model(**batch)
#         loss = criterion(out.logits, batch["labels"])
#         loss.backward()
#         optimizer.step()
#         scheduler.step()
#         total_loss += loss.item()
#     val_metrics = evaluate(val_loader)
#     print(f"Epoch {epoch+1}/{EPOCHS} | TrainLoss={total_loss/len(train_loader):.4f} | Val F1={val_metrics['f1']:.4f}")
#     if val_metrics["f1"] > best_f1:
#         best_f1 = val_metrics["f1"]
#         patience_ctr = 0
#         model.save_pretrained(OUTPUT_DIR)
#         tokenizer.save_pretrained(OUTPUT_DIR)
#     else:
#         patience_ctr += 1
#         if patience_ctr >= patience:
#             print("Early stopping triggered.")
#             break

# # -----------------------
# # Evaluate best model
# # -----------------------
# model = BertForSequenceClassification.from_pretrained(OUTPUT_DIR).to(device)
# print("\nValidation metrics:", evaluate(val_loader))
# print("Test metrics:", evaluate(test_loader))
# print(f"\n✅ Saved best model to {OUTPUT_DIR}")

# # -----------------------
# # Inference Helper
# # -----------------------
# def predict_texts(texts):
#     model.eval()
#     enc = tokenizer(texts, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LENGTH).to(device)
#     with torch.no_grad():
#         out = model(**enc)
#         probs = out.logits.softmax(dim=-1).cpu().numpy()
#         preds = probs.argmax(axis=-1)
#     return [
#         {"text":t,"prob_non_fraud":float(p[0]),"prob_fraud":float(p[1]),
#          "pred_label":["Non_Fraudulent","Fraudulent"][int(pred)]}
#         for t,p,pred in zip(texts,probs,preds)
#     ]

# print("\nSample predictions:")
# for r in predict_texts([
#     "your account will be blocked unless you verify now",
#     "hi, your package will be delivered tomorrow"
# ]):
#     print(r)


In [ ]:
# # ===============================
# # CELL 4 — MFCC Hybrid (Keras 3 safe; no Lambda)
# # ===============================
# !pip -q install -U librosa==0.10.1 soundfile==0.12.1

# import os, librosa, numpy as np, pandas as pd, tensorflow as tf
# from tensorflow import keras
# from tensorflow.keras import layers, models, callbacks, optimizers
# from sklearn.utils.class_weight import compute_class_weight
# from sklearn.metrics import classification_report, confusion_matrix

# print("TensorFlow:", tf.__version__)
# print("Keras:", keras.__version__)
# print("GPU devices:", tf.config.list_physical_devices("GPU"))

# # -----------------------
# # Config
# # -----------------------
# SR = 16000
# N_MFCC = 40
# MAX_LEN = 200
# BATCH = 32
# EPOCHS = 50
# SEED = 42

# CLEAN_DIR = "/kaggle/working/clean"
# TRAIN_CSV = f"{CLEAN_DIR}/train.csv"
# VAL_CSV   = f"{CLEAN_DIR}/val.csv"
# TEST_CSV  = f"{CLEAN_DIR}/test.csv"
# OUT_PATH  = "/kaggle/working/mfcc_cnn.keras"

# np.random.seed(SEED)
# tf.random.set_seed(SEED)

# # -----------------------
# # Load CSVs
# # -----------------------
# df_train = pd.read_csv(TRAIN_CSV)
# df_val   = pd.read_csv(VAL_CSV)
# df_test  = pd.read_csv(TEST_CSV)
# print(f"Loaded train={len(df_train)}, val={len(df_val)}, test={len(df_test)}")

# # -----------------------
# # MFCC extraction (robust)
# # -----------------------
# def extract_mfcc(path, sr=SR, n_mfcc=N_MFCC, max_len=MAX_LEN):
#     try:
#         y, _ = librosa.load(path, sr=sr)
#         if len(y) < sr * 0.1:
#             return np.zeros((max_len, n_mfcc * 3), dtype=np.float32)

#         mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
#         # robust delta width for very short signals
#         width = min(9, max(3, (mfcc.shape[1] // 2) * 2 + 1))
#         delta = librosa.feature.delta(mfcc, width=width)
#         delta2 = librosa.feature.delta(mfcc, order=2, width=width)

#         feat = np.vstack([mfcc, delta, delta2]).T  # (T, 120)
#         if feat.shape[0] < max_len:
#             feat = np.pad(feat, ((0, max_len - feat.shape[0]), (0, 0)))
#         feat = feat[:max_len, :]
#         return feat.astype(np.float32)
#     except Exception as e:
#         print(f"[WARN] MFCC failed for {path}: {e}")
#         return np.zeros((max_len, n_mfcc * 3), dtype=np.float32)

# def build_dataset(df):
#     X, y = [], []
#     for p, label in zip(df["audio_path_clean"], df["label_num"]):
#         X.append(extract_mfcc(p))
#         y.append(int(label))
#     X = np.expand_dims(np.array(X, np.float32), -1)  # (N, 200, 120, 1)
#     y = keras.utils.to_categorical(y, 2)
#     return X, y

# print("Extracting MFCC features...")
# X_train, y_train = build_dataset(df_train)
# X_val, y_val = build_dataset(df_val)
# X_test, y_test = build_dataset(df_test)
# print("Shapes →", X_train.shape, y_train.shape)

# # -----------------------
# # Custom Attention layer (Keras 3 safe; serializable)
# # -----------------------
# class TemporalAttention(layers.Layer):
#     def __init__(self, **kwargs):
#         super().__init__(**kwargs)

#     def build(self, input_shape):
#         d = int(input_shape[-1])
#         self.w1 = self.add_weight(shape=(d, d), initializer="glorot_uniform", name="attn_w1")
#         self.b1 = self.add_weight(shape=(d,), initializer="zeros", name="attn_b1")
#         self.w2 = self.add_weight(shape=(d, 1), initializer="glorot_uniform", name="attn_w2")
#         self.b2 = self.add_weight(shape=(1,), initializer="zeros", name="attn_b2")
#         super().build(input_shape)

#     def call(self, x):
#         # x: (batch, time, dim)
#         h = tf.tanh(tf.linalg.matmul(x, self.w1) + self.b1)   # (B,T,D)
#         e = tf.linalg.matmul(h, self.w2) + self.b2            # (B,T,1)
#         a = tf.nn.softmax(e, axis=1)                          # (B,T,1)
#         ctx = tf.reduce_sum(x * a, axis=1)                    # (B,D)
#         return ctx

#     def get_config(self):
#         return super().get_config()

# # -----------------------
# # Model
# # -----------------------
# def build_model(input_shape=(MAX_LEN, N_MFCC * 3, 1)):
#     inp = layers.Input(shape=input_shape)
#     x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inp)
#     x = layers.BatchNormalization()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
#     x = layers.Dropout(0.2)(x)

#     x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
#     x = layers.Dropout(0.3)(x)

#     x = layers.Conv2D(128, (3, 3), activation="relu", padding="same")(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
#     x = layers.Dropout(0.3)(x)

#     # reshape for LSTM
#     # shapes: (None, 200,120,1) -> after 3x pool: (None, 25, 15, 128)
#     t = x.shape[1]
#     f = x.shape[2]
#     c = x.shape[3]
#     x = layers.Reshape((t, f * c))(x)  # (None, 25, 15*128)

#     x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
#     x = layers.Dropout(0.4)(x)

#     attn = TemporalAttention()(x)  # (None, 256) because BiLSTM(128) returns 256 features

#     x = layers.Dense(256, activation="relu")(attn)
#     x = layers.Dropout(0.4)(x)
#     out = layers.Dense(2, activation="softmax")(x)

#     model = models.Model(inp, out)
#     model.compile(
#         optimizer=optimizers.Adam(learning_rate=5e-4),
#         loss="categorical_crossentropy",
#         metrics=["accuracy"]
#     )
#     return model

# model = build_model()
# model.summary()

# # -----------------------
# # Training
# # -----------------------
# cw = compute_class_weight("balanced", classes=np.unique(df_train["label_num"]), y=df_train["label_num"])
# cw_dict = dict(enumerate(cw))
# print("Class weights:", cw_dict)

# cb = [
#     callbacks.ModelCheckpoint(OUT_PATH, save_best_only=True, monitor="val_accuracy", mode="max", verbose=1),
#     callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1)
# ]

# hist = model.fit(
#     X_train, y_train,
#     validation_data=(X_val, y_val),
#     epochs=EPOCHS,
#     batch_size=BATCH,
#     class_weight=cw_dict,
#     callbacks=cb,
#     verbose=1
# )

# # -----------------------
# # Evaluate (clean load; no Lambda; register custom layer)
# # -----------------------
# from tensorflow import keras as tfk

# model = tfk.models.load_model(
#     OUT_PATH,
#     custom_objects={"TemporalAttention": TemporalAttention}
# )

# pred = model.predict(X_test, verbose=0)
# y_true, y_pred = np.argmax(y_test, 1), np.argmax(pred, 1)

# print("\nTest Classification Report:")
# print(classification_report(y_true, y_pred, target_names=["Non_Fraudulent", "Fraudulent"]))
# print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

# pd.DataFrame(
#     classification_report(
#         y_true, y_pred, target_names=["Non_Fraudulent", "Fraudulent"], output_dict=True
#     )
# ).T.to_csv("/kaggle/working/mfcc_metrics.csv")

# print("\nModel saved →", OUT_PATH)


In [ ]:
# # ===============================
# # CELL 5 — Spectrogram CNNs (DenseNet169, EfficientNet-B3/B4)
# # ===============================
# !pip -q install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 tqdm timm==1.0.3 librosa==0.10.1 soundfile==0.12.1

# import os, gc, random, torch, librosa, numpy as np, pandas as pd, soundfile as sf
# from pathlib import Path
# from tqdm import tqdm
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader
# import timm
# from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# # -----------------------
# # Config
# # -----------------------
# SR = 16000
# N_MELS = 128
# IMG_SIZE = 224
# BATCH = 16
# EPOCHS = 15
# LR = 1e-4
# SEED = 42

# np.random.seed(SEED)
# random.seed(SEED)
# torch.manual_seed(SEED)
# torch.cuda.manual_seed_all(SEED)

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# print("✅ Device:", DEVICE)

# ROOT = Path("/kaggle/working/clean")
# TRAIN_CSV = ROOT / "train.csv"
# VAL_CSV = ROOT / "val.csv"
# TEST_CSV = ROOT / "test.csv"

# OUT_DIR = Path("/kaggle/working/spec_models")
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# # -----------------------
# # Data Prep
# # -----------------------
# def load_audio(path):
#     try:
#         y, _ = librosa.load(path, sr=SR)
#         if len(y) == 0:
#             return np.zeros(SR)
#         return y
#     except Exception as e:
#         print(f"[WARN] Failed to load {path}: {e}")
#         return np.zeros(SR)

# def make_mel(y):
#     mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS)
#     mel_db = librosa.power_to_db(mel, ref=np.max)
#     mel_db = np.clip((mel_db + 80) / 80, 0, 1)
#     return mel_db.astype(np.float32)

# def spec_to_img(spec):
#     spec = np.stack([spec, spec, spec], axis=-1)  # (H, W, 3)
#     spec = torch.tensor(spec).permute(2, 0, 1).unsqueeze(0)  # (1, 3, H, W)
#     spec = F.interpolate(spec, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
#     return spec.squeeze(0)  # (3, 224, 224)

# # -----------------------
# # Dataset with SpecAugment
# # -----------------------
# class SpecDataset(Dataset):
#     def __init__(self, df, train=True):
#         self.df = df.reset_index(drop=True)
#         self.train = train

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         y = load_audio(row["audio_path_clean"])
#         spec = make_mel(y)
#         img = spec_to_img(spec)
#         label = torch.tensor(int(row["label_num"])).long()
#         if self.train:
#             img = self.augment(img)
#         return img, label

#     def augment(self, img):
#         img = img.clone()
#         # Frequency masking
#         if random.random() < 0.5:
#             num_mask = random.randint(1, 2)
#             for _ in range(num_mask):
#                 f = random.randint(0, 16)
#                 f0 = random.randint(0, N_MELS - f)
#                 img[:, f0:f0+f, :] = 0
#         # Time masking
#         if random.random() < 0.5:
#             t = random.randint(0, 30)
#             t0 = random.randint(0, IMG_SIZE - t)
#             img[:, :, t0:t0+t] = 0
#         return img

# # -----------------------
# # Data Loaders
# # -----------------------
# df_train = pd.read_csv(TRAIN_CSV)
# df_val   = pd.read_csv(VAL_CSV)
# df_test  = pd.read_csv(TEST_CSV)

# train_dl = DataLoader(SpecDataset(df_train, train=True), batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
# val_dl   = DataLoader(SpecDataset(df_val, train=False), batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
# test_dl  = DataLoader(SpecDataset(df_test, train=False), batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

# # -----------------------
# # Label smoothing
# # -----------------------
# class LabelSmoothingLoss(nn.Module):
#     def __init__(self, smoothing=0.1):
#         super().__init__()
#         self.smoothing = smoothing

#     def forward(self, preds, targets):
#         confidence = 1.0 - self.smoothing
#         logprobs = F.log_softmax(preds, dim=-1)
#         nll = -logprobs.gather(dim=-1, index=targets.unsqueeze(1)).squeeze(1)
#         smooth = -logprobs.mean(dim=-1)
#         return (confidence * nll + self.smoothing * smooth).mean()

# criterion = LabelSmoothingLoss(0.1)

# # -----------------------
# # Train/Eval helpers
# # -----------------------
# def train_one_epoch(model, loader, optimizer):
#     model.train()
#     total, correct, loss_sum = 0, 0, 0
#     for xb, yb in loader:
#         xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
#         optimizer.zero_grad(set_to_none=True)
#         out = model(xb)
#         loss = criterion(out, yb)
#         loss.backward()
#         optimizer.step()
#         total += yb.size(0)
#         correct += (out.argmax(1) == yb).sum().item()
#         loss_sum += loss.item() * yb.size(0)
#     return loss_sum / total, correct / total

# @torch.no_grad()
# def evaluate(model, loader):
#     model.eval()
#     total, correct, loss_sum = 0, 0, 0
#     preds_all, labels_all = [], []
#     for xb, yb in loader:
#         xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
#         out = model(xb)
#         loss = criterion(out, yb)
#         total += yb.size(0)
#         correct += (out.argmax(1) == yb).sum().item()
#         loss_sum += loss.item() * yb.size(0)
#         preds_all.extend(out.argmax(1).cpu().numpy())
#         labels_all.extend(yb.cpu().numpy())
#     return loss_sum / total, correct / total, preds_all, labels_all

# # -----------------------
# # Model training
# # -----------------------
# def run_model(name, model_fn):
#     print(f"\n🚀 Training {name} ...")
#     model = model_fn(pretrained=True)
#     if hasattr(model, "classifier") and isinstance(model.classifier, nn.Linear):
#         model.classifier = nn.Linear(model.classifier.in_features, 2)
#     elif hasattr(model, "classifier") and isinstance(model.classifier, nn.Sequential):
#         in_feat = model.classifier[-1].in_features
#         model.classifier = nn.Linear(in_feat, 2)
#     elif hasattr(model, "head"):
#         in_feat = model.get_classifier().in_features
#         model.reset_classifier(2)
#     model.to(DEVICE)

#     optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
#     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)
#     best_acc, best_path = 0.0, OUT_DIR / f"best_{name}.pth"

#     for epoch in range(EPOCHS):
#         tr_loss, tr_acc = train_one_epoch(model, train_dl, optimizer)
#         val_loss, val_acc, _, _ = evaluate(model, val_dl)
#         scheduler.step(val_loss)
#         print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train {tr_acc:.3f} | Val {val_acc:.3f}")
#         if val_acc > best_acc:
#             best_acc = val_acc
#             torch.save(model.state_dict(), best_path)
#     print(f"✅ Best Val Acc: {best_acc:.3f} | Saved → {best_path}")
#     return best_path

# model_defs = {
#     "densenet169": lambda pretrained=True: timm.create_model("densenet169", pretrained=pretrained, num_classes=2),
#     "efficientnet_b3": lambda pretrained=True: timm.create_model("efficientnet_b3", pretrained=pretrained, num_classes=2),
#     "efficientnet_b4": lambda pretrained=True: timm.create_model("efficientnet_b4", pretrained=pretrained, num_classes=2)
# }

# paths = {}
# for name, fn in model_defs.items():
#     paths[name] = run_model(name, fn)
#     gc.collect()
#     torch.cuda.empty_cache()

# # -----------------------
# # Ensemble Evaluation
# # -----------------------
# @torch.no_grad()
# def ensemble_predict(paths, loader):
#     models_list = []
#     for n, p in paths.items():
#         model = model_defs[n](pretrained=False)
#         model.load_state_dict(torch.load(p, map_location=DEVICE))
#         model.to(DEVICE).eval()
#         models_list.append(model)

#     preds_all, labels_all = [], []
#     for xb, yb in loader:
#         xb = xb.to(DEVICE)
#         outs = [F.softmax(m(xb), dim=-1) for m in models_list]
#         out_mean = torch.stack(outs).mean(0)
#         preds_all.extend(out_mean.argmax(1).cpu().numpy())
#         labels_all.extend(yb.numpy())
#     return labels_all, preds_all

# labels, preds = ensemble_predict(paths, test_dl)
# print("\n✅ Ensemble Test Report:")
# print(classification_report(labels, preds, target_names=["Non_Fraudulent", "Fraudulent"]))
# print("Confusion matrix:\n", confusion_matrix(labels, preds))

# pd.DataFrame(
#     classification_report(labels, preds, target_names=["Non_Fraudulent","Fraudulent"], output_dict=True)
# ).T.to_csv("/kaggle/working/spec_ensemble_metrics.csv")

# print("\n🧾 All models saved in:", OUT_DIR)


In [ ]:
# ===============================
# CELL — Fusion Ensemble (BERT + MFCC + Spectrogram)
# ===============================
!pip -q install -U "transformers==4.41.2" "torch>=2.3.0" timm==1.0.3 librosa==0.10.1 soundfile==0.12.1 scikit-learn==1.6.1 tensorflow==2.18.0 keras==3.4.1

import os, torch, librosa, numpy as np, pandas as pd
import torch.nn.functional as F
from pathlib import Path
from transformers import BertTokenizerFast, BertForSequenceClassification
from tensorflow import keras
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import timm
from tensorflow.keras import layers

# -----------------------
# Paths
# -----------------------
ROOT_MODELS = Path("/kaggle/input/fraud-fusion-models")
CLEAN_DIR = Path("/kaggle/working/clean")
TEST_CSV = CLEAN_DIR / "test.csv"

BERT_DIR = ROOT_MODELS / "Bert_finetuned"
MFCC_MODEL_PATH = ROOT_MODELS / "mfcc_cnn.keras"
SPEC_DIR = ROOT_MODELS / "spec_models"

df_test = pd.read_csv(TEST_CSV)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ Device:", device, "| Test samples:", len(df_test))

# -----------------------
# 1️⃣ BERT predictions
# -----------------------
print("🔹 Running BERT...")
# ✅ Load BERT locally from Kaggle dataset folder
bert_model = BertForSequenceClassification.from_pretrained(
    str(BERT_DIR),
    local_files_only=True
).to(device)

bert_tokenizer = BertTokenizerFast.from_pretrained(
    str(BERT_DIR),
    local_files_only=True
)


def bert_predict_prob(texts):
    enc = bert_tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        out = bert_model(**enc)
        probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
    return probs[:,1]

bert_probs = []
for t in df_test["transcription_clean"].astype(str).tolist():
    if len(t.strip()) < 3:
        bert_probs.append(0.5)
    else:
        bert_probs.append(bert_predict_prob([t])[0])

df_test["bert_prob"] = bert_probs
print("✅ BERT done")

# -----------------------
# 2️⃣ MFCC predictions
# -----------------------
@tf.keras.utils.register_keras_serializable(package="Custom")
class TemporalAttention(layers.Layer):
    """
    Alternate training-time variant using add_weight (attn_w1, attn_b1, attn_w2, attn_b2)
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        d = int(input_shape[-1])
        self.attn_w1 = self.add_weight(shape=(d, d), initializer="glorot_uniform", name="attn_w1")
        self.attn_b1 = self.add_weight(shape=(d,), initializer="zeros", name="attn_b1")
        self.attn_w2 = self.add_weight(shape=(d, 1), initializer="glorot_uniform", name="attn_w2")
        self.attn_b2 = self.add_weight(shape=(1,), initializer="zeros", name="attn_b2")
        super().build(input_shape)

    def call(self, x):
        h = tf.tanh(tf.linalg.matmul(x, self.attn_w1) + self.attn_b1)  # (B,T,D)
        e = tf.linalg.matmul(h, self.attn_w2) + self.attn_b2           # (B,T,1)
        a = tf.nn.softmax(e, axis=1)                                   # (B,T,1)
        return tf.reduce_sum(x * a, axis=1)                            # (B,D)

    def get_config(self):
        return super().get_config()



keras_model = keras.models.load_model(
    MFCC_MODEL_PATH,
    custom_objects={"TemporalAttention": TemporalAttention},
    safe_mode=False
)

SR, N_MFCC, MAX_LEN = 16000, 40, 200

def extract_mfcc_feat(path):
    y, _ = librosa.load(path, sr=SR)
    mfcc = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=N_MFCC)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat = np.vstack([mfcc, delta, delta2]).T
    if feat.shape[0] < MAX_LEN:
        feat = np.pad(feat, ((0, MAX_LEN - feat.shape[0]), (0, 0)))
    else:
        feat = feat[:MAX_LEN, :]
    feat = np.expand_dims(feat, axis=(0, -1))
    return feat

mfcc_probs = []
for p in df_test["audio_path_clean"]:
    f = extract_mfcc_feat(p)
    mfcc_probs.append(keras_model.predict(f, verbose=0)[0,1])

df_test["mfcc_prob"] = mfcc_probs
print("✅ MFCC done")

# -----------------------
# 3️⃣ Spectrogram ensemble
# -----------------------
print("🔹 Running Spectrogram ensemble...")

def make_mel(path):
    y, _ = librosa.load(path, sr=16000)
    mel = librosa.feature.melspectrogram(y=y, sr=16000, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = np.clip((mel_db + 80)/80, 0, 1)
    mel_db = np.stack([mel_db]*3, axis=0)
    mel_db = torch.tensor(mel_db).unsqueeze(0)
    mel_db = F.interpolate(mel_db, size=(224,224), mode="bilinear", align_corners=False)
    return mel_db.to(device)

def load_spec_model(name, path):
    model = timm.create_model(name, pretrained=False, num_classes=2)
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval().to(device)
    return model

models_spec = {
    "densenet169": load_spec_model("densenet169", SPEC_DIR/"best_densenet169.pth"),
    "efficientnet_b3": load_spec_model("efficientnet_b3", SPEC_DIR/"best_efficientnet_b3.pth"),
    "efficientnet_b4": load_spec_model("efficientnet_b4", SPEC_DIR/"best_efficientnet_b4.pth"),
}

spec_probs = []
for p in df_test["audio_path_clean"]:
    mel = make_mel(p)
    preds = []
    for m in models_spec.values():
        with torch.no_grad():
            o = m(mel)
            preds.append(F.softmax(o, dim=-1)[:,1].cpu().numpy()[0])
    spec_probs.append(float(np.mean(preds)))

df_test["spec_prob"] = spec_probs
print("✅ Spectrogram done")

# -----------------------
# 4️⃣ Fusion model (Logistic Regression)
# -----------------------
X = df_test[["bert_prob","mfcc_prob","spec_prob"]].values
y = df_test["label_num"].values

fusion = LogisticRegression(max_iter=1000)
fusion.fit(X, y)
pred = fusion.predict(X)
prob = fusion.predict_proba(X)[:,1]

print("\n✅ Fusion Ensemble Report:")
print(classification_report(y, pred, target_names=["Non_Fraudulent","Fraudulent"]))
print("Confusion matrix:\n", confusion_matrix(y, pred))
print("AUC:", roc_auc_score(y, prob))

pd.DataFrame(
    classification_report(y, pred, target_names=["Non_Fraudulent","Fraudulent"], output_dict=True)
).T.to_csv("/kaggle/working/fusion_metrics.csv")

print("\n🧾 Fusion metrics saved → /kaggle/working/fusion_metrics.csv")


In [ ]:
fusion_df = df_test[["audio_path_clean", "label_num", "bert_prob", "mfcc_prob", "spec_prob"]]
fusion_df.to_csv("/kaggle/working/fusion_features.csv", index=False)
print("✅ Saved fusion feature file → /kaggle/working/fusion_features.csv")


In [ ]:
# ===============================
# CELL — Compare Fusion Meta-Models (CV + Save Best)
# ===============================
import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from joblib import dump

# Optional models
HAS_XGB = True
HAS_LGB = True
try:
    import xgboost as xgb
except Exception:
    HAS_XGB = False
try:
    import lightgbm as lgb
except Exception:
    HAS_LGB = False

ROOT = Path("/kaggle/working")
CSV_FUSION = ROOT / "fusion_features.csv"   # preferred if you saved earlier
CLEAN_TEST = ROOT / "clean" / "test.csv"    # fallback if probs exist here
OUT_CSV = ROOT / "fusion_model_comparison.csv"
BEST_PATH = ROOT / "best_fusion_model.pkl"
BEST_META  = ROOT / "best_fusion_meta.json"
FIG_PNG = ROOT / "fusion_model_comparison.png"
FIG_PDF = ROOT / "fusion_model_comparison.pdf"

# -----------------------
# Load features
# -----------------------
if CSV_FUSION.exists():
    df = pd.read_csv(CSV_FUSION)
else:
    df = pd.read_csv(CLEAN_TEST)
    required = {"bert_prob","mfcc_prob","spec_prob","label_num"}
    if not required.issubset(df.columns):
        raise FileNotFoundError(
            "Need fusion_features.csv or test.csv with ['bert_prob','mfcc_prob','spec_prob','label_num']."
        )

X = df[["bert_prob","mfcc_prob","spec_prob"]].values.astype(np.float32)
y = df["label_num"].astype(int).values
print(f"Data: {X.shape}, Fraudulent: {y.sum()}, Non-Fraudulent: {(y==0).sum()}")

# -----------------------
# Fusion candidates
# -----------------------
def make_models():
    models = []

    # 1) Simple average (no training)
    models.append(("SimpleAvg", None))

    # 2) Weighted averages (grid) — choose best by CV
    weight_grid = [
        (0.50,0.25,0.25),
        (0.60,0.20,0.20),
        (0.50,0.40,0.10),
        (0.40,0.30,0.30),
        (0.33,0.33,0.34),
        (0.30,0.40,0.30),
        (0.30,0.30,0.40),
        (0.25,0.25,0.50),
        (0.25,0.50,0.25),
        (0.20,0.40,0.40),
    ]
    # use a placeholder name; we’ll evaluate inside CV and report the best
    models.append(("WeightedAvgGrid", weight_grid))

    # 3) Logistic Regression (strong baseline)
    models.append(("LogReg", LogisticRegression(max_iter=500)))

    # 4) MLP (small)
    mlp = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(hidden_layer_sizes=(16,8), activation="relu", alpha=1e-3,
                              max_iter=1000, random_state=42))
    ])
    models.append(("MLP", mlp))

    # 5) XGBoost (if available)
    if HAS_XGB:
        xgbm = xgb.XGBClassifier(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            n_jobs=4
        )
        models.append(("XGBoost", xgbm))

    # 6) LightGBM (if available)
    if HAS_LGB:
        lgbm = lgb.LGBMClassifier(
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=3,
            num_leaves=7,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_alpha=0.5,
            reg_lambda=1.0,
            random_state=42,
            verbose=-1,
        )
        models.append(("LightGBM", lgbm))

    return models

# -----------------------
# Cross-validation
# -----------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

records = []
summary = {}

def eval_metrics(y_true, prob):
    pred = (prob > 0.5).astype(int)
    return (
        accuracy_score(y_true, pred),
        f1_score(y_true, pred, average="weighted"),
        roc_auc_score(y_true, prob)
    )

def run_cv_for_weight_grid(X, y, weight_grid):
    # choose the best weight triple by mean F1 across folds
    grid_scores = []
    for w in weight_grid:
        accs, f1s, aucs = [], [], []
        for tr, va in skf.split(X, y):
            p = (X[va] * np.array(w)).sum(axis=1)  # weighted average of probs
            acc, f1, auc = eval_metrics(y[va], p)
            accs.append(acc); f1s.append(f1); aucs.append(auc)
        grid_scores.append((w, np.mean(accs), np.mean(f1s), np.mean(aucs)))
    # pick best by F1
    best = sorted(grid_scores, key=lambda t: t[2], reverse=True)[0]
    return best, grid_scores

models = make_models()
best_weighted = None
per_fold_rows = []

for name, mdl in models:
    if name == "SimpleAvg":
        accs, f1s, aucs = [], [], []
        for tr, va in skf.split(X, y):
            p = X[va].mean(axis=1)
            acc, f1, auc = eval_metrics(y[va], p)
            accs.append(acc); f1s.append(f1); aucs.append(auc)
            per_fold_rows.append({"model": name, "fold_acc": acc, "fold_f1": f1, "fold_auc": auc})
        summary[name] = {"acc_mean": np.mean(accs), "acc_std": np.std(accs),
                         "f1_mean": np.mean(f1s), "f1_std": np.std(f1s),
                         "auc_mean": np.mean(aucs), "auc_std": np.std(aucs)}

    elif name == "WeightedAvgGrid":
        best, grid_all = run_cv_for_weight_grid(X, y, mdl)
        w, acc_m, f1_m, auc_m = best
        best_weighted = {"weights": w, "acc_mean": acc_m, "f1_mean": f1_m, "auc_mean": auc_m}
        # store like others with std from best weights
        accs, f1s, aucs = [], [], []
        for tr, va in skf.split(X, y):
            p = (X[va] * np.array(w)).sum(axis=1)
            acc, f1, auc = eval_metrics(y[va], p)
            accs.append(acc); f1s.append(f1); aucs.append(auc)
            per_fold_rows.append({"model": f"WeightedAvg{w}", "fold_acc": acc, "fold_f1": f1, "fold_auc": auc})
        summary[f"WeightedAvg{w}"] = {"acc_mean": np.mean(accs), "acc_std": np.std(accs),
                                      "f1_mean": np.mean(f1s), "f1_std": np.std(f1s),
                                      "auc_mean": np.mean(aucs), "auc_std": np.std(aucs)}

    else:
        accs, f1s, aucs = [], [], []
        for tr, va in skf.split(X, y):
            Xtr, Xva = X[tr], X[va]; ytr, yva = y[tr], y[va]
            model = mdl
            # re-instantiate for each fold to avoid leakage
            if name == "LogReg":
                model = LogisticRegression(max_iter=500)
            elif name == "MLP":
                model = Pipeline([
                    ("scaler", StandardScaler()),
                    ("mlp", MLPClassifier(hidden_layer_sizes=(16,8), activation="relu",
                                          alpha=1e-3, max_iter=1000, random_state=42))
                ])
            elif name == "XGBoost" and HAS_XGB:
                model = xgb.XGBClassifier(
                    n_estimators=400, learning_rate=0.05, max_depth=3,
                    subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=4
                )
            elif name == "LightGBM" and HAS_LGB:
                model = lgb.LGBMClassifier(
                    n_estimators=1000, learning_rate=0.03, max_depth=3,
                    num_leaves=7, subsample=0.9, colsample_bytree=0.9,
                    reg_alpha=0.5, reg_lambda=1.0, random_state=42, verbose=-1
                )
            model.fit(Xtr, ytr)
            p = model.predict_proba(Xva)[:,1]
            acc, f1, auc = eval_metrics(y[va], p)
            accs.append(acc); f1s.append(f1); aucs.append(auc)
            per_fold_rows.append({"model": name, "fold_acc": acc, "fold_f1": f1, "fold_auc": auc})
        summary[name] = {"acc_mean": np.mean(accs), "acc_std": np.std(accs),
                         "f1_mean": np.mean(f1s), "f1_std": np.std(f1s),
                         "auc_mean": np.mean(aucs), "auc_std": np.std(aucs)}

# -----------------------
# Summarize + rank
# -----------------------
rows = []
for k,v in summary.items():
    rows.append([k, v["acc_mean"], v["acc_std"], v["f1_mean"], v["f1_std"], v["auc_mean"], v["auc_std"]])
sum_df = pd.DataFrame(rows, columns=["Model","Acc_mean","Acc_std","F1_mean","F1_std","AUC_mean","AUC_std"])
sum_df = sum_df.sort_values("F1_mean", ascending=False).reset_index(drop=True)
sum_df.to_csv(OUT_CSV, index=False)
print("\nFusion CV summary (5-fold):")
print(sum_df)

# -----------------------
# Pick best and save
# -----------------------
best_name = sum_df.iloc[0]["Model"]
best_meta = {"best_model": best_name}

if best_name.startswith("WeightedAvg"):
    # parse weights from the model name "WeightedAvg(w1, w2, w3)"
    w = best_name.replace("WeightedAvg","").strip()
    w = w.strip("()")
    w = tuple(float(x) for x in w.split(","))
    best_meta["type"] = "weighted_average"
    best_meta["weights"] = w
    # save weights only
    dump({"type": "weighted_average", "weights": w}, BEST_PATH)
else:
    # train best learner on all data and save
    if best_name == "LogReg":
        best_model = LogisticRegression(max_iter=500).fit(X, y)
    elif best_name == "MLP":
        best_model = Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPClassifier(hidden_layer_sizes=(16,8), activation="relu",
                                  alpha=1e-3, max_iter=1000, random_state=42))
        ]).fit(X, y)
    elif best_name == "XGBoost" and HAS_XGB:
        best_model = xgb.XGBClassifier(
            n_estimators=400, learning_rate=0.05, max_depth=3,
            subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=4
        ).fit(X, y)
    elif best_name == "LightGBM" and HAS_LGB:
        best_model = lgb.LGBMClassifier(
            n_estimators=1000, learning_rate=0.03, max_depth=3,
            num_leaves=7, subsample=0.9, colsample_bytree=0.9,
            reg_alpha=0.5, reg_lambda=1.0, random_state=42, verbose=-1
        ).fit(X, y)
    else:
        # fallback
        best_model = LogisticRegression(max_iter=500).fit(X, y)
        best_name = "LogReg"
    dump(best_model, BEST_PATH)
    best_meta["type"] = "sklearn"
    best_meta["sklearn_estimator"] = best_name

with open(BEST_META, "w") as f:
    json.dump(best_meta, f, indent=2)
print(f"\nSaved best fusion → {BEST_PATH}")
print(f"Meta → {BEST_META}")

# -----------------------
# Quick figure (F1 mean ± std)
# -----------------------
labels = sum_df["Model"].tolist()
means  = sum_df["F1_mean"].values
errs   = sum_df["F1_std"].values

plt.figure(figsize=(8,4.5))
x = np.arange(len(labels))
plt.bar(x, means, yerr=errs, capsize=5)
plt.xticks(x, labels, rotation=20, ha="right")
plt.ylabel("F1 (weighted)")
plt.title("Fusion Meta-Model Comparison (5-fold CV)")
plt.tight_layout()
plt.savefig(FIG_PNG, dpi=300)
plt.savefig(FIG_PDF)
plt.show()

print(f"\nSaved summary CSV → {OUT_CSV}")
print(f"Saved figure → {FIG_PNG} / {FIG_PDF}")


In [ ]:
# ===============================
# CELL — Fusion CV (Show All Weighted Averages)
# ===============================
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# use existing fusion_features.csv
df = pd.read_csv("/kaggle/working/fusion_features.csv")
X = df[["bert_prob","mfcc_prob","spec_prob"]].values
y = df["label_num"].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Weight grid
weight_grid = [
    (0.50,0.25,0.25),
    (0.60,0.20,0.20),
    (0.50,0.40,0.10),
    (0.40,0.30,0.30),
    (0.33,0.33,0.34),
    (0.30,0.40,0.30),
    (0.30,0.30,0.40),
    (0.25,0.25,0.50),
    (0.25,0.50,0.25),
    (0.20,0.40,0.40),
]

def eval_metrics(y_true, prob):
    pred = (prob > 0.5).astype(int)
    acc = accuracy_score(y_true, pred)
    f1  = f1_score(y_true, pred, average="weighted")
    auc = roc_auc_score(y_true, prob)
    return acc, f1, auc

results = []
for w in weight_grid:
    accs, f1s, aucs = [], [], []
    for tr, va in skf.split(X, y):
        p = (X[va] * np.array(w)).sum(axis=1)
        acc, f1, auc = eval_metrics(y[va], p)
        accs.append(acc); f1s.append(f1); aucs.append(auc)
    results.append({
        "Weights": w,
        "Acc_mean": np.mean(accs),
        "Acc_std": np.std(accs),
        "F1_mean": np.mean(f1s),
        "F1_std": np.std(f1s),
        "AUC_mean": np.mean(aucs),
        "AUC_std": np.std(aucs)
    })

results_df = pd.DataFrame(results).sort_values("F1_mean", ascending=False).reset_index(drop=True)

print("🎯 Weighted Average Grid Results (5-Fold CV):")
print(results_df.round(5))

# save as CSV for record
results_df.to_csv("/kaggle/working/weighted_avg_grid_results.csv", index=False)
print("\n📁 Saved → /kaggle/working/weighted_avg_grid_results.csv")
print(f"✅ Best weights: {tuple(results_df.iloc[0]['Weights'])} with F1={results_df.iloc[0]['F1_mean']:.4f}")


In [ ]:
# ===============================
# FRAUD FUSION API — DEBUG MODE
# ===============================
!pip -q install gradio "transformers==4.41.2" "torch>=2.3.0" "timm==1.0.3" "librosa==0.10.1" "soundfile==0.12.1" "tensorflow==2.18.0" "keras==3.4.1"

import gradio as gr
import torch, librosa, numpy as np, json, requests, re, io
import torch.nn.functional as F
from contextlib import redirect_stdout

from pathlib import Path
from transformers import BertTokenizerFast, BertForSequenceClassification
from tensorflow import keras
from tensorflow.keras import layers
import timm, tensorflow as tf

# -----------------------
# Paths
# -----------------------
ROOT_MODELS = Path("/kaggle/input/fraud-fusion-models")
BERT_DIR = ROOT_MODELS / "Bert_finetuned"
MFCC_MODEL_PATH = ROOT_MODELS / "mfcc_cnn.keras"
SPEC_DIR = ROOT_MODELS / "spec_models"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ Device:", device)

# ===============================
# Load BERT
# ===============================
bert_model = BertForSequenceClassification.from_pretrained(str(BERT_DIR), local_files_only=True).to(device)
bert_tokenizer = BertTokenizerFast.from_pretrained(str(BERT_DIR), local_files_only=True)

def predict_bert_prob(text):
    if not text or len(text.strip()) < 2:
        return 0.5
    enc = bert_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        out = bert_model(**enc)
        probs = torch.softmax(out.logits, dim=-1).cpu().numpy()[0]
    return float(probs[1])  # Fraudulent prob


# ===============================
# Load MFCC CNN with TemporalAttention
# ===============================
@keras.utils.register_keras_serializable(package="Custom")
class TemporalAttention(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    def build(self, input_shape):
        d = int(input_shape[-1])
        self.attn_w1 = self.add_weight(shape=(d, d), initializer="glorot_uniform", name="attn_w1")
        self.attn_b1 = self.add_weight(shape=(d,), initializer="zeros", name="attn_b1")
        self.attn_w2 = self.add_weight(shape=(d, 1), initializer="glorot_uniform", name="attn_w2")
        self.attn_b2 = self.add_weight(shape=(1,), initializer="zeros", name="attn_b2")
        super().build(input_shape)
    def call(self, x):
        h = tf.tanh(tf.linalg.matmul(x, self.attn_w1) + self.attn_b1)
        e = tf.linalg.matmul(h, self.attn_w2) + self.attn_b2
        a = tf.nn.softmax(e, axis=1)
        return tf.reduce_sum(x * a, axis=1)
    def get_config(self): return super().get_config()

keras_model = keras.models.load_model(
    MFCC_MODEL_PATH,
    custom_objects={"TemporalAttention": TemporalAttention},
    safe_mode=False
)

def predict_mfcc_prob(mfcc_array):
    feat = np.array(mfcc_array, dtype=np.float32)
    if feat.ndim == 2:
        feat = np.expand_dims(feat, axis=(0, -1))
    return float(keras_model.predict(feat, verbose=0)[0, 1])


# ===============================
# Load Spectrogram Ensemble
# ===============================
def load_spec_model(name, path):
    model = timm.create_model(name, pretrained=False, num_classes=2)
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval().to(device)
    return model

models_spec = {
    "densenet169": load_spec_model("densenet169", SPEC_DIR / "best_densenet169.pth"),
    "efficientnet_b3": load_spec_model("efficientnet_b3", SPEC_DIR / "best_efficientnet_b3.pth"),
    "efficientnet_b4": load_spec_model("efficientnet_b4", SPEC_DIR / "best_efficientnet_b4.pth"),
}

def predict_spec_prob(spec_array):
    spec = np.array(spec_array, dtype=np.float32)
    if spec.ndim == 2:
        spec = np.stack([spec, spec, spec], axis=0)
    spec = torch.tensor(spec).unsqueeze(0)
    spec = F.interpolate(spec, size=(224, 224), mode="bilinear", align_corners=False).to(device)
    preds = []
    with torch.no_grad():
        for m in models_spec.values():
            o = m(spec)
            preds.append(F.softmax(o, dim=-1)[:, 1].cpu().numpy()[0])
    return float(np.mean(preds))


# ===============================
# Fusion (DEBUG ENABLED)
# ===============================
BEST_WEIGHTS = (0.6, 0.2, 0.2)

def fusion_predict(text, mfcc_json, spec_json):
    # print("\n🛰️ Incoming Request ===============================")
    # print("TEXT:", text[:150] if text else "<empty>")
    # print("MFCC (first 2 rows):", str(mfcc_json)[:250])
    # print("SPEC (first 2 rows):", str(spec_json)[:250])
    # print("===================================================")

    try:
        # Try parsing inputs
        if isinstance(mfcc_json, str): mfcc = json.loads(mfcc_json)
        else: mfcc = mfcc_json
        if isinstance(spec_json, str): spec = json.loads(spec_json)
        else: spec = spec_json

        # Individual model probabilities
        bert_p = predict_bert_prob(text)
        mfcc_p = predict_mfcc_prob(mfcc)
        spec_p = predict_spec_prob(spec)

        # Weighted fusion
        final_p = BEST_WEIGHTS[0]*bert_p + BEST_WEIGHTS[1]*mfcc_p + BEST_WEIGHTS[2]*spec_p
        label = "Fraudulent" if final_p >= 0.5 else "Non-Fraudulent"
        # print("Label => ", label, " Probability => ", round(final_p, 4), "=>", round(bert_p, 4), round(mfcc_p, 4), round(spec_p, 4), sep=" ")
        result = {
            "label": label,
            "final_prob": round(final_p, 4),
            "bert_prob": round(bert_p, 4),
            "mfcc_prob": round(mfcc_p, 4),
            "spec_prob": round(spec_p, 4),
            "debug_raw_input": {
                "text": text,
                "mfcc_shape": np.shape(mfcc),
                "spec_shape": np.shape(spec)
            }
        }

        # print("✅ Returning:", result)
        # print("===================================================\n")
        return result

    except Exception as e:
        print("❌ Exception:", str(e))
        return {"error": str(e)}


# # ===============================
# # Gradio Interface (Server)
# # ===============================
# demo = gr.Interface(
#     fn=fusion_predict,
#     inputs=[
#         gr.Textbox(label="Text (Transcription)"),
#         gr.Textbox(label="MFCC Array (JSON or list)"),
#         gr.Textbox(label="Spectrogram Array (JSON or list)")
#     ],
#     outputs=[gr.JSON(label="Prediction")],
#     title="🎧 Fraud Detection Fusion API (Debug Mode)",
#     description="Logs all input payloads and returns them for debugging.",
#     allow_flagging="never"
# )
# demo.queue(concurrency_count=None, max_size=None, status_update_rate=None, enabled=False)


# # -------------------------------
# # Launch and inspect endpoints
# # -------------------------------
# demo.launch(share=True)

In [ ]:
import gradio as gr
print("✅ Gradio version:", gr.__version__)


In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## 🎧 Fraud Detection Fusion API (Debug Mode)")

    txt = gr.Textbox(label="Text (Transcription)")
    mfcc = gr.Textbox(label="MFCC Array (JSON or list)")
    spec = gr.Textbox(label="Spectrogram Array (JSON or list)")
    out = gr.JSON(label="Prediction")

    submit = gr.Button("Run Fusion")
    submit.click(fusion_predict, inputs=[txt, mfcc, spec], outputs=out, api_name="fusion_predict")

# ==============================================================
#  4️⃣ Launch
# ==============================================================

if __name__ == "__main__":
    demo.launch(share=True)

In [ ]:
import requests, json, numpy as np

mfcc = np.random.randn(200, 120).tolist()
spec = np.random.randn(224, 224).tolist()

url = "https://87e10a69365bf286f7.gradio.live/gradio_api/api/fusion_predict"
payload = {"data": ["KYC expired please verify now", json.dumps(mfcc), json.dumps(spec)]}

r = requests.post(url, json=payload)
print(r.json())
